# Notebook 5 : Réduction de dimension

Notebook préparé par [Chloé-Agathe Azencott](http://cazencott.info) et modifié par [Victor Laigle](https://eaglev-sci.github.io/).

Dans ce notebook il s'agit d'explorer plusieurs techniques de réduction de dimension.

In [ ]:
# charger numpy, padnas et matplotlib (avec les alias np, pd et plt)
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
plt.rc('font', **{'size': 12}) # règle la taille de police globalement pour les plots (en pt)

## 1. Analyse en composantes principales

Dans cette section, nous allons effectuer une analyse en composantes principales d'un jeu de données décrivant les scores obtenus par les meilleurs athlètes ayant participé en 2004 à une épreuve de décathlon, aux Jeux Olympiques d'Athènes ou au Décastar de Talence.

### Chargement des données

Les données sont contenues dans le fichier `decathlon.txt`.

Le fichier contient 42 lignes et 13 colonnes.

La première ligne est un en-tête qui décrit les contenus des colonnes.

Les lignes suivantes décrivent les 41 athlètes.

Les 10 premières colonnes contiennent les scores obtenus aux différentes épreuves.
La 11ème colonne contient le classement.
La 12ème colonne contient le nombre de points obtenus.
La 13ème colonne contient une variable qualitative qui précise l'épreuve (JO ou Décastar) concernée.

Nous allons examiner ces données avec la librairie `pandas`.

In [ ]:
my_data = pd.read_csv('data/decathlon.txt', sep="\t")  # lire les données dans un dataframe

__Alternativement :__ Si vous avez besoin de télécharger le fichier (par exemple sur colab) :

In [ ]:
# !wget https://raw.githubusercontent.com/ThomasWalter/CourseFoundationsML/SIA2025/Notebooks/5-Reduction/data/decathlon.txt

# my_data = pd.read_csv('decathlon.txt', sep="\t") 

In [ ]:
my_data.head()

### Visualisation

Une __matrice de nuages de points__ est une visualisation en k x k panneaux des relations deux à deux entre k variables :
* sur la diagonale, l'histogramme pour chacune des variables 
* hors diagonale, les nuages de points entre deux variables (non standardisées).

https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.plotting.scatter_matrix.html

Par exemple, avec toutes les données numériques :

In [ ]:
from pandas.plotting import scatter_matrix

scatter_matrix(my_data, alpha=0.5, s=60, figsize=(18, 18));  # alpha = transparence des points, s = taille des points

On peut aussi restreindre la figure à certaines variables :

In [ ]:
scatter_matrix(my_data[['Shot.put','High.jump', '400m']], alpha=0.5, s=60, figsize=(8, 8));

Alternativement, la librairie `seaborn` permet des visualisations plus élaborées que `matplotlib`. Vous pouvez par exemple explorer les capacités de `jointplot`. 
https://seaborn.pydata.org/generated/seaborn.jointplot.html

On affiche, pour une paire de variables, un nuage de points accompagné des histogrammes représentant la distribution de chacune des variables. On peut également ajouter une estimation de la densité des deux variables et une droite de régression linéaire.

In [ ]:
import seaborn as sns
sns.set_style('whitegrid')

sns.jointplot(x='Shot.put', y='400m', data = my_data, 
              height=6, space=0, color='b')

sns.jointplot(x='Shot.put', y='Discus', data = my_data, 
              kind='reg', height=6, space=0, color='b');

## ACP

Nous allons maintenant effectuer une analyse en composantes principales des scores aux 10 épreuves.

Commençons par extraire les données :

In [ ]:
X = np.array(my_data.drop(columns=['Points', 'Rank', 'Competition']))
print(X.shape)

### Standardisation des données

In [ ]:
from sklearn import preprocessing

In [ ]:
std_scaler = preprocessing.StandardScaler().fit(X)
X_scaled = std_scaler.transform(X)

### Calcul des composantes principales

Les algorithmes de factorisation de matrice de `scikit-learn` sont inclus dans le module `decomposition`. Pour  l'ACP, référez-vous à : 
http://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html

In [ ]:
from sklearn import decomposition

Remarque : Nous avons ici peu de variables et pouvons nous permettre de calculer toutes les PC. 

La plupart des algorithmes implémentés dans `scikit-learn` suivent le fonctionnement suivant : 
* on instancie un objet, correspondant à un type d'algorithme/modèle, avec ses hyperparamètres (ici le nombre de composantes)
* on utilise la méthode `fit` pour passer les données à cet algorithme
* les paramètres appris sont maintenant accessibles comme arguments de cet objet.

In [ ]:
# Instanciation d'un objet PCA pour 10 composantes principales
pca = decomposition.PCA(n_components=10)

In [ ]:
# On passe maintenant les données standardisées à cet objet
# C'est ici que se font les calculs
pca.fit(X_scaled)

### Proportion de variance expliquée par les PCs

Nous allons maintenant afficher la proportion de variance expliquée par les différentes composantes. Il est accessible dans le paramètre `explained_variance_ration_` de notre objet `pca`.

In [ ]:
plt.plot(np.arange(1, 11), pca.explained_variance_ratio_, marker='o')

plt.xlabel("Composantes principales")
plt.ylabel("Proportion de variance expliquée");

Nous pouvons aussi afficher la proportion *cumulative* de variance expliquée

In [ ]:
np.cumsum(pca.explained_variance_ratio_)

In [ ]:
plt.plot(np.arange(1, 11), np.cumsum(pca.explained_variance_ratio_), marker='o')

plt.ylim(0, 1.05)
plt.xlabel("Nombre de composantes principales")
plt.ylabel("Proportion (cumulative) de variance expliquée");
plt.title("Proportion cumulative de variance expliquée");

On peut aussi calculer la proportion de variance expliquée par les 4 (par exemple) premières PC avec

In [ ]:
print("%.2f" % np.sum(pca.explained_variance_ratio_[:4]))

__Questions :__ 
* Quelle est la proportion de variance expliquée par les deux premières composantes ? 
* Combien de composantes faudrait-il utiliser pour expliquer 80% de la variance des données ?

**Réponses :**

- Sur le graphique montrant la proportion cumulative de variance expliquée par les composantes, on peut voir que les deux premières composantes principales expliquent environ 50% de la variance totale des données. On peut également calculer cette valeur avec le code suivant, qui montre qu'on a exactement 50%de variance expliquée par les deux premières composantes :

In [ ]:
print("%.2f" % np.sum(pca.explained_variance_ratio_[:2]))

- Le graphique montre aussi qu'il faut 5 composantes principales pour expliquer 80% de la variance des données. Cela signifie qu'on peut réduire la dimension des données de 10 à 5 tout en conservant une grande partie de l'information contenue dans les données initiales.

### Projection des données sur les deux premières composantes principales

Nous allons maintenant utiliser uniquement les deux premières composantes principales.

Commençons par calculer la nouvelle représentation des données, c'est-à-dire leur projection sur ces deux PC. Autrement dit, on recalcule les coordonnées des points (les athlètes) dans le nouvel espace de dimension 2 représenté par les deux premières composantes principales. La coordonnée d'un point sur une composante principale est donnée par le produit scalaire entre le vecteur des coordonnées du point dans l'espace initial et le vecteur propre correspondant à cette composante principale. 

In [ ]:
pca = decomposition.PCA(n_components=2)
pca.fit(X_scaled)
X_projected = pca.transform(X_scaled)
print(X_projected.shape)

On peut afficher un nuage de points représentant les données selon ces deux PC.

In [ ]:
fig = plt.figure(figsize=(5, 5))

plt.scatter(X_projected[:, 0], X_projected[:, 1])

plt.xlabel("PC 1")
plt.ylabel("PC 2");

En tant que tel, le graphique ne nous apprend pas grand-chose. On peut cependant essayer de voir si les composantes principales correspondent à des caractéristiques que l'on peut interpréter.

On peut colorier chaque point du nuage de points ci-dessus en fonction du classement de l'athlète qu'il représente.

In [ ]:
fig = plt.figure(figsize=(5, 5))

plt.scatter(X_projected[:, 0], X_projected[:, 1], c=my_data['Rank'])
# plt.scatter(X_projected[:, 0], X_projected[:, 1], c=my_data['Points'])

plt.xlabel("PC 1")
plt.ylabel("PC 2")
plt.colorbar(label='classement');
# plt.colorbar(label='points');

**Question :** Qu'en conclure sur l'interprétation de la PC1 ?

**Réponse :** On observe ici que la PC1 est très corrélée au classement des athlètes. Pour des valeurs faibles de la PC1, on observe des valeurs faibles du classement (donc un bon classement), et à l'inverse, pour des valeurs élevées de la PC1, des classements moins bons. La PC1 semble donc représenter la performance globale des athlètes au décathlon. C'est encore plus clair si on utilise la variable 'Points' pour colorier les points : on observe une corrélation presque parfaite de la PC1 avec le nombre de points obtenus par les athlètes (beaucoup de points pour des valeurs faibles de la PC1, moins de points pour des valeurs élevées de la PC1). La différence entre les rangs et les points s'explique par la troisième variable qualitative (JO ou Décastar) : pour obtenir un classement équivalent, les athlètes participant aux JO doivent marquer plus de points que ceux participant au Décastar, du fait du niveau de compétition plus élevé aux JO.

> A noter : il est important de regarder dans quel sens chacune des composantes principales est corrélée avec ce qu'elle explique. Ici, de manière peu intuitive, les basses valeurs de la PC1 correspondent à de meilleures performances.

### Interprétation des deux composantes principales
Chaque composante principale est une combinaison linéaire des variables décrivant les données. Les poids de cette combinaison linéaire sont accesibles dans `pca.components_`.

Nous pouvons visualiser non pas les individus, mais les 10 variables dans l'espace des 2 composantes principales.

In [ ]:
pcs = pca.components_
print(pcs[0])

In [ ]:
fig = plt.figure(figsize=(6, 6))

plt.scatter(pcs[0], pcs[1])
for (x_coordinate, y_coordinate, feature_name) in zip(pcs[0], pcs[1], my_data.columns[:10]):
    plt.text(x_coordinate, y_coordinate, feature_name)                          
    
plt.xlabel("Corrélation à la PC1")
plt.ylabel("Corrélation à la PC2");

__Question :__ Quelles variables ont des corrélations très similaires aux deux composantes principales ? Qu'en déduire sur leur similarité ?

__Réponse :__ 

On peut observer que certaines variables se retrouvent dans la même "zone" du graphique.En particulier, les variables "100m" et "110m hurdles" ont des corrélations très similaires aux deux premières composantes principales, avec des valeurs élevées sur la PC1 et faibles sur la PC2. Les variables "Shot put" et "Discus" ont également des corrélations similaires, avec des valeurs négatives élevées sur la PC1 et plutôt positives et élevées sur la PC2.

On peut donc en déduire que ces paires de variables sont similaires entre elles, c'est-à-dire qu'elles mesurent des aspects similaires de la performance des athlètes au décathlon. En se reportant à la matrice en nuages de points de la partie visualisation vue plus haut dans le notebook, on peut vérifier cette similarité : on observe en effet que les variables "100m" et "110m hurdles" sont fortement corrélées entre elles, tout comme les variables "Shot put" et "Discus". 

A l'inverse, les variables "Long jump" et "400m", opposées sur ce graphique, sont corrélées négativement entre elles comme on peut l'observer sur le nuage de points correspondant dans la partie visualisation.

__Question :__ Comment interpréter le signe des corrélations des variables à la première composantes principales ?

__Réponse :__ 

On peut d'abord observer que les variables "100m", "400m" et "110m hurdles" ont toutes une corrélation positive élevée avec la PC1, tandis que les variables "Long.jump", "Discus", "Shot put" et "High.jump" sont corrélées négativement avec cette même PC1. Cela est cohérent avec la nature des épreuves : les trois premières sont des épreuves de course où l'on cherche à avoir un temps de course le plus faible possible, tandis que les autres sont des épreuves de saut ou de lancer où la performance est mesurée par la distance, longueur ou hauteur qui doivent être les plus élevées possibles. 

Le signe des corrélations des variables à la première composante principale doit être interprété en parallèle de notre interprétation précédente de la PC1. On a vu que la PC1 représentait la performance globale des athlètes, avec des valeurs négative pour une bonne performance et des valeurs positives correspondant à une performance moins bonne. Cela est tout à fait cohérent avec les signes de corrélations observés ici : pour les épreuves mentionnées, une valeur élevée pour une épreuve de course correspond à une mauvaise performance et est donc corrélée positivement avec la PC1, tandis qu'une valeur élevée pour une épreuve de saut ou de lancer correspond à une bonne performance et est donc corrélée négativement avec la PC1.

> A noter également : 
> - les composantes principales sont définies au signe près. Il aurait donc été tout à fait possible que la PC1 soit définie dans l'autre sens, avec des corrélations également inversées. D'où l'importance d'interpréter les corrélations en parallèle de l'interprétation des composantes principales elles-mêmes.
> - il est possible que dans un jeu de données réel, les composantes principales ne soient pas aussi facilement interprétables que dans cet exemple, qu'il faille utilisé une combinaison de composantes pour trouver une interprétation ou que l'interprétation soit plus claire sur une autre composante que la ou les première(s).
> - les composantes principales capturent la variance maximale de nos données, ce qui aide à différencier les individus / observations. Dans cet exemple, on peut donc en déduire que la différence au classement s'est jouée principalement sur les épreuves de course (100m, 400m, 110m hurdles) et sur les épreuves de saut et lancer (Long jump, Discus, Shot put, High jump). Cela suggère aussi que les athlètes se sont moins différenciés les uns des autres sur les autres épreuves ayant une faible corrélation avec la première composante principale.


## 2. Données « Olivetti »

Nous allons maintenant utiliser la réduction de dimension pour représenter en deux dimensions un jeu de données contenant des visages. Il s'agit d'un jeu de données classique, contenant 400 photos de 64 par 64 pixels. Il s'agit de photos des visages de 40 personnes différentes (10 photos par personne), étiquetées par un numéro de classe entre 0 et 39 identifiant la personne.

Nous pouvons charger ce jeu de données directement grâce à scikit-learn :

In [ ]:
from sklearn import datasets

In [ ]:
data = datasets.fetch_olivetti_faces()

__Si vous n'arrivez pas à télécharger les données :__
* Aller sur : https://github.com/CroncLee/PCA-face-recognition/blob/master/olivetti_py3.pkz
* Télécharger le fichier (bouton Download) 
utiliser la commande
```
    data = datasets.fetch_olivetti_faces(data_home="<PATH TO DATA>")
```
En remplaçant <PATH TO DATA> par le chemin vers le dossier où vous avez enregistré les données.

In [ ]:
X = data.data
y = data.target

In [ ]:
print(X.shape)

In [ ]:
print("Les données contiennent %d classes" % len(np.unique(y)))

Chaque image est représentée par une valeur (niveau de gris) pour chacun de ses pixels. 

Nous pouvons visualiser ces images à condition de réorganiser ces valeurs (= un vecteur de longueur 4096) en matrices 64x64. Par exemle ci-dessous pour l'image à l'index 190 :

In [ ]:
plt.imshow(X[190, :].reshape((64, 64)), interpolation='nearest', cmap=plt.cm.gray)
plt.colorbar();

### PCA

Commençons par une analyse en composantes principales comme à la section précédente :

In [ ]:
pca = decomposition.PCA(n_components=2)
X_transformed_pca = pca.fit_transform(X)

Chaque image est maintenant représentée par non pas 4096 variables, mais par deux variables. Nous pouvons les visualiser en nuage de point, et les colorer par classe :

In [ ]:
plt.scatter(X_transformed_pca[:, 0], X_transformed_pca[:, 1], 
            c=y, cmap=plt.get_cmap('tab20'))
plt.xlabel("Première composante principale")
plt.ylabel("Deuxième composante principale");

> Note: On utilise ici une palette de couleurs (tab20) contenant seulement 20 couleurs distinctes, alors qu'on a 40 classes. Certaines classes auront donc la même couleur. 

__Question :__ les images du même visage (= de la même classe) ont-elles des représentations proches ?

__Réponse :__ On observe globalement un rapprochement des points appartenant à la même classe (même couleur dans le graphique), donc les représentations des images d'une même classe sont relativement proches. Cela dit, on est tout de même loin d'avoir des groupes nettement séparés. Cela signifie donc que l'appartenance à une classe n'est pas parfaitement capturée par les deux premières composantes principales, possiblement parce que ce n'est pas la principale source de variabilité dans nos données. 

Nous pouvons visualiser la contribution de chaque pixel à la première composante principale :

In [ ]:
plt.imshow(pca.components_[0, :].reshape((64,64)), interpolation='nearest', cmap=plt.cm.gray);

Puis la contribution de chaque pixel à la deuxième composante principale :

In [ ]:
plt.imshow(pca.components_[1, :].reshape((64,64)), interpolation='nearest', cmap=plt.cm.gray);

__Question :__ 
Quelle est la proportion de variance expliquée par les deux premières composantes ? 

__Réponse :__
Comme vu plus haut, on calcule la proportion de variance expliquée par les deux premières composantes principales avec le code suivant :

In [ ]:
print("%.2f" % np.sum(pca.explained_variance_ratio_[:2]))

__Question :__
En observant les 2 premières composantes principales, que remarquez-vous sur les zones ou les structures mises en avant dans l'image ? Que peut-on en déduire sur les informations qu'elles capturent ?

__Réponse :__
On observe des choses différentes sur les deux composantes principales :
- La première composante principale met en avant certaines parties du visage, en particulier les contours du nez et des yeux, les pommettes ou encore la formes des lunettes que l'on peut identifier. Ce sont donc, d'après notre algorithme, des zones comportant des variations importantes dans les images, qui sont donc utiles pour différencier les visages entre eux, et cela paraît tout à fait cohérent avec l'intuition que l'on peut avoir sur notre façon de reconnaître et différencier des visages. 
- La deuxième composante principale capture des variations selon un gradient gauche-droite sur l'image, ce qui peut correspondre à des variations d'éclairage et à des zones d'ombre observées sur les visages. On capture donc de l'information qui n'est pas directement liée à l'identité de la personne.

### tSNE

Essayons une autre méthode de réduction de dimension pour mieux séparer nos classes.

Nous allons maintenant utiliser la même démarche, mais avec tSNE, grâce à la classe [TSNE](https://scikit-learn.org/stable/modules/generated/sklearn.manifold.TSNE.html) du module `manifold`.

Maintenant, nous allons :

1. Créer l'objet tsne à partir de la classe citée ci-dessus. Ici, nous voulons réduire notre ensemble de données à deux dimensions afin de le visualiser.
2. Assigner les informations de notre ensemble de données à cet objet.
3. Transformer nos données pour obtenir de nouvelles coordonnées en deux dimensions.

In [ ]:
from sklearn import manifold

In [ ]:
tsne = manifold.TSNE(n_components=2, init='random', learning_rate='auto')

In [ ]:
X_transformed = tsne.fit_transform(X)

In [ ]:
plt.scatter(X_transformed[:, 0], X_transformed[:, 1], 
            c=y, cmap=plt.get_cmap('tab20'))
plt.xlabel("Première composante tSNE")
plt.ylabel("Deuxième composante tSNE");

On observe déjà une meilleure séparation des classes qu'avec l'ACP, même si certaines classes ont l'air d'être séparées en plusieurs groupes.

#### Influence du paramètre de perplexité

Le principal hyperparamètre influençant la représentation obtenue par l'algorithme tSNE est le paramètre « perplexité ». Celui-ci représente le nombre de voisins pour lesquels les distances sont préservées. Il influence donc la préservation de la structure locale (perplexité faible) ou globale (perplexité élevée). La représentation obtenue peut varier considérablement en fonction de ce paramètre.

Nous allons donc maintenant observer l'influence de ce paramètre de perplexité sur la visualisation obtenue avec tSNE. 

Identifiez la valeur par défaut du paramètre utilisée dans le premier appel à la fonction TSNE. Testez ensuite différentes valeurs du paramètre de perplexité et affichez les résultats correspondants.

Par exemple :

1. Refaire une visualisation tSNE avec une perplexité de 1.
2. Refaire une visualisation tSNE avec une perplexité de 100.
3. Comparer les résultats obtenus.

In [ ]:
tsne = manifold.TSNE(n_components=2, init='random', learning_rate='auto', perplexity=1)

In [ ]:
X_transformed = tsne.fit_transform(X)

In [ ]:
plt.scatter(X_transformed[:, 0], X_transformed[:, 1], 
            c=y, cmap=plt.get_cmap('tab20'))
plt.xlabel("Première composante tSNE")
plt.ylabel("Deuxième composante tSNE");

In [ ]:
tsne = manifold.TSNE(n_components=2, init='random', learning_rate='auto', perplexity=100)

In [ ]:
X_transformed = tsne.fit_transform(X)

In [ ]:
plt.scatter(X_transformed[:, 0], X_transformed[:, 1], 
            c=y, cmap=plt.get_cmap('tab20'))
plt.xlabel("Première composante tSNE")
plt.ylabel("Deuxième composante tSNE");

__Question :__ Comment les résultats de la visualisation t-SNE changent-ils lorsque vous modifiez la perplexité ? Dans le premier graphique, que vaut le paramètre perplexity ?

__Réponse :__
- La documentation nous indique la valeur par défaut du paramètre de perplexité, utilisée dans le premier graphique : `perplexity=30`.
- Lorsque la perplexité est faible (par exemple 1), la visualisation t-SNE met davantage l'accent sur la structure locale des données. Des points très proches dans l'espace initial (à 4096 dimensions) restent très proches dans la visualisation en 2D, mais le reste de la structure globale est complètement perdu, ce qui se traduit par des petits groupes très serrés et répartis un peu partout dans l'espace 2D.
- Avec une perplexité plus élevée (100 dans notre exemple), l'algorithme t-SNE capture davantage la structure globale des données. On retrouve une séparation plus claire des points appartenant à une même classe, mais on perd une partie de la précision locale : les points proches dans l'espace initial ne sont plus aussi bien localisés (ou regroupés) à un même endroit et on a maintenant des points appartenant à une même classe qui sont plus dispersés qu'avec une perplexité plus faible.

Globalement, l'algorithme t-SNE paraît plus efficace que l'ACP pour séparer les différentes classes de ce jeu de données, et la perplexité par défaut, 30, semble être un bon compromis entre préservation de les structures locales et globales du dataset. Attention cependant, la visualisation t-SNE souffre de certains défauts :
- elle n'est pas aussi interprétable que l'ACP, car on ne sait pas comment les composantes sont construites à partir des variables initiales
- les distances entre les groupes ne sont pas forcément représentatives des distances dans l'espace initial

## Conclusion

Nous sommes arrivés à la fin de ce notebook. Voici un résumé de ce que nous avons couvert, avec les points clés :

- Nous avons utilisé la bibliothèque `scikit-learn` pour explorer plusieurs techniques de **réduction de dimension**, permettant de comprendre et visualiser les données en utilisant moins de variables.

- Nous avons commencé avec l'**Analyse en Composantes Principales (ACP)** sur un jeu de données de décathlon contenant 10 épreuves. L'ACP calcule des combinaisons linéaires orthogonales des variables originales appelées **composantes principales**, chacune expliquant une portion décroissante de la variance des données.

- Nous avons montré l'importance de **standardiser les données** avant d'appliquer l'ACP, car les composantes principales sont sensibles à l'échelle des variables. Cela garantit que chaque variable contribue équitablement.

- Nous avons analysé la **proportion de variance expliquée** par chaque composante principale et sa forme **cumulative**. Cela permet de déterminer combien de composantes retenir pour conserver une certaine quantité d'information (par exemple 80%).

- Nous avons visualisé les données projetées sur les deux premières composantes principales et coloré les points selon le classement des athlètes. Cela démontre comment l'ACP révèle des structures dans les données et permet l'**interprétation** : les variables avec des contributions similaires sont fortement corrélées.

- Nous avons utilisé l'ACP sur le **jeu de données Olivetti faces**, contenant 400 images de visages (64×64 pixels, 4096 variables par image). Malgré la réduction de 4096 à 2 dimensions, les images du même visage apparaissent groupées, montrant que l'ACP capture les variations majeures entre les individus.

- Nous avons visualisé les **composantes principales elles-mêmes** (en les remettant en forme d'images), révélant que la première composante capture les variations globales de luminosité du visage, tandis que la deuxième capture d'autres structures spatiales.

- Nous avons exploré **t-SNE** (t-Distributed Stochastic Neighbor Embedding), une technique de réduction de dimension non-linéaire qui préserve mieux la **structure locale** des données, contrairement à l'ACP qui préserve la variance globale.

- Nous avons démontré l'influence du **paramètre de perplexité** de t-SNE : une perplexité faible préserve la structure locale fine mais peut fragmenter les données, tandis qu'une perplexité élevée préserve davantage la structure globale. Ce paramètre offre un compromis entre la préservation des voisinages locaux et globaux.

- Enfin, nous avons comparé les résultats : **l'ACP** offre une réduction de dimension interprétable et linéaire, idéale pour l'exploration et l'analyse; **t-SNE** offre une meilleure séparation visuelle des classes en 2D grâce à sa nature non-linéaire, mais est moins interprétable et plus coûteuse en calcul.